# Zenith Traffic CO₂ Estimator

Open in Google Colab. Select **Runtime → Change runtime type → T4 GPU** if available, then **Runtime → Run all**. CPU also works, with slower inference. The default run downloads a 21-second highway clip and produces an annotated video, a CO₂ breakdown, and a downloadable ZIP.

Pretrained YOLO11 detects vehicles and ByteTrack follows them. The emissions model multiplies each completed traversal by the road segment length and a vehicle-class factor in g CO₂/km. Sample factors and the 50 m demo segment are illustrative. [Sample video](https://supervision.roboflow.com/0.30.9/assets/) · [CCTV emissions study](https://journal.uin-alauddin.ac.id/index.php/instek/article/view/73033)

In [ ]:
# @title 1. Settings
USE_DEMO_VIDEO = True # @param {type:"boolean"}
VIDEO_PATH = "/content/traffic.mp4" # @param {type:"string"}
SEGMENT_METRES = 0.0 # @param {type:"number"}
GATE_A_Y = 0.39 # @param {type:"number"}
GATE_B_Y = 0.61 # @param {type:"number"}
GATE_X_MIN = 0.0 # @param {type:"number"}
GATE_X_MAX = 1.0 # @param {type:"number"}
CAR_G_PER_KM = -1.0 # @param {type:"number"}
MOTORCYCLE_G_PER_KM = -1.0 # @param {type:"number"}
BUS_G_PER_KM = -1.0 # @param {type:"number"}
TRUCK_G_PER_KM = -1.0 # @param {type:"number"}
FACTOR_SOURCE = "" # @param {type:"string"}
MAX_VIDEO_SECONDS = 30 # @param {type:"integer"}
AUTO_DOWNLOAD = True # @param {type:"boolean"}

For a custom video, upload it through Colab's **Files** panel, set its path, and turn off **USE_DEMO_VIDEO**. Set the two horizontal gates across your lanes using fractions of the frame measured from the top left. Enter the measured road distance between the gates, four local vehicle factors, and their source. Check the preview before processing. Set the duration cap to zero for the full video.

Use a fixed camera where vehicles travel roughly straight through both gates. Both directions share the same measured distance. Narrow the horizontal limits to a single carriageway if the distances differ.

In [ ]:
# @title 2. Install dependencies
%pip -q install "ultralytics>=8.3,<9" "lap>=0.5.12"

In [ ]:
# @title 3. Load video and preview gates
import math
import shutil
import tempfile
from pathlib import Path
from urllib.request import urlopen

import cv2
import torch
from IPython.display import Image, Video, display

DEMO_URL = "https://media.roboflow.com/supervision/video-examples/vehicles.mp4"
CLASSES = {2: "car", 3: "motorcycle", 5: "bus", 7: "truck"}
GATES = {"A": GATE_A_Y, "B": GATE_B_Y}
FONT = cv2.FONT_HERSHEY_SIMPLEX
GATE_COLORS = {"A": (70, 230, 255), "B": (255, 210, 90)}

if not 0 < GATE_A_Y < GATE_B_Y < 1 or not 0 <= GATE_X_MIN < GATE_X_MAX <= 1:
    raise ValueError("Set gate A above gate B and use frame coordinates between 0 and 1.")
if not isinstance(MAX_VIDEO_SECONDS, int) or MAX_VIDEO_SECONDS < 0:
    raise ValueError("Choose a nonnegative whole number for MAX_VIDEO_SECONDS.")

ROOT = Path("/content/zenith_traffic_co2")
ROOT.mkdir(parents=True, exist_ok=True)
if USE_DEMO_VIDEO:
    video_path = ROOT / "roboflow_vehicles.mp4"
    segment_metres = 50.0
    factors = {"car": 160.0, "motorcycle": 65.0, "bus": 800.0, "truck": 900.0}
    factor_source = "Illustrative demo factors and assumed 50 m segment"
    if not video_path.exists():
        partial = video_path.with_suffix(".part")
        try:
            with urlopen(DEMO_URL, timeout=90) as remote, partial.open("wb") as local:
                shutil.copyfileobj(remote, local)
            partial.replace(video_path)
        except OSError as error:
            partial.unlink(missing_ok=True)
            raise RuntimeError("Could not download the sample. Retry or upload your own video.") from error
else:
    video_path = Path(VIDEO_PATH).expanduser()
    segment_metres = SEGMENT_METRES
    factors = {"car": CAR_G_PER_KM, "motorcycle": MOTORCYCLE_G_PER_KM,
               "bus": BUS_G_PER_KM, "truck": TRUCK_G_PER_KM}
    factor_source = FACTOR_SOURCE.strip()
    if not math.isfinite(segment_metres) or segment_metres <= 0:
        raise ValueError("Enter the measured, positive road distance in SEGMENT_METRES.")
    if any(not math.isfinite(value) or value < 0 for value in factors.values()) or not factor_source:
        raise ValueError("Enter four nonnegative fleet factors and their source.")
if not video_path.is_file():
    raise FileNotFoundError(f"Upload your video and check its path: {video_path}")

def draw_gates(frame):
    h, w = frame.shape[:2]
    for gate, y in GATES.items():
        start = (round(GATE_X_MIN * (w - 1)), round(y * (h - 1)))
        end = (round(GATE_X_MAX * (w - 1)), start[1])
        cv2.line(frame, start, end, GATE_COLORS[gate], 2)
        cv2.putText(frame, gate, (start[0] + 8, max(15, start[1] - 7)), FONT, 0.7,
                    GATE_COLORS[gate], 2, cv2.LINE_AA)

capture = cv2.VideoCapture(str(video_path))
try:
    ok, preview = capture.read()
finally:
    capture.release()
if not ok:
    raise ValueError(f"OpenCV could not decode: {video_path}")
scale = min(1, 960 / preview.shape[1])
if scale < 1:
    preview = cv2.resize(preview, None, fx=scale, fy=scale)
draw_gates(preview)
ok, encoded = cv2.imencode(".jpg", preview)
if ok:
    display(Image(data=encoded.tobytes()))
print(f"{video_path.name} | segment {segment_metres:g} m | {factor_source}")

In [ ]:
# @title 4. Define the segment counter and CO₂ model
from collections import Counter
from dataclasses import dataclass, field

@dataclass
class Track:
    last: tuple | None = None
    first_gate: tuple | None = None
    votes: Counter = field(default_factory=Counter)

class SegmentEstimator:
    def __init__(self, gates, x_limits, metres, factors):
        self.gates = gates
        self.x_limits = x_limits
        self.metres = metres
        self.factors = factors
        self.tracks = {}
        self.counted = set()
        self.events = []
        self.total_g = 0.0

    def observe(self, track_id, label, x, y, seconds):
        if track_id in self.counted:
            return
        track = self.tracks.setdefault(track_id, Track())
        track.votes[label] += 1
        previous, track.last = track.last, (x, y, seconds)
        if previous is None:
            return
        px, py, pt = previous
        if seconds - pt > 0.5:
            track.first_gate = None
            track.votes = Counter({label: 1})
            return
        if y == py:
            return
        downward = y > py
        crossings = []
        left, right = self.x_limits
        for gate, gate_y in self.gates.items():
            crossed = (py < gate_y <= y) if downward else (y <= gate_y < py)
            if crossed:
                fraction = (gate_y - py) / (y - py)
                if left <= px + fraction * (x - px) <= right:
                    crossings.append((fraction, gate))
        direction = "A_to_B" if downward else "B_to_A"
        start_gate = "A" if downward else "B"
        for fraction, gate in sorted(crossings):
            crossing_time = pt + fraction * (seconds - pt)
            if gate == start_gate:
                track.first_gate = (direction, crossing_time)
            elif track.first_gate and track.first_gate[0] == direction:
                label = track.votes.most_common(1)[0][0]
                factor = self.factors[label]
                grams = self.metres / 1000 * factor
                self.events.append({
                    "track_id": track_id, "vehicle_class": label, "direction": direction,
                    "entry_seconds": round(track.first_gate[1], 3),
                    "exit_seconds": round(crossing_time, 3), "segment_metres": self.metres,
                    "factor_g_co2_per_km": factor, "estimated_co2_g": round(grams, 6),
                })
                self.total_g += grams
                self.counted.add(track_id)
                del self.tracks[track_id]
                return
            else:
                track.first_gate = None

In [ ]:
# @title 5. Detect and track vehicles
import subprocess
from tqdm.auto import tqdm
from ultralytics import YOLO

def run_video(path, output, estimator, max_seconds):
    model = YOLO("yolo11n.pt")
    device = 0 if torch.cuda.is_available() else "cpu"
    capture = cv2.VideoCapture(str(path))
    writer = None
    try:
        fps = capture.get(cv2.CAP_PROP_FPS)
        if not math.isfinite(fps) or fps <= 0:
            raise ValueError("Could not read the frame rate. Re-export as a constant-FPS MP4.")
        frame_count = int(capture.get(cv2.CAP_PROP_FRAME_COUNT))
        limit = max(1, round(max_seconds * fps)) if max_seconds else None
        stride = max(1, round(fps / 12))
        duration = min(frame_count, limit) if frame_count and limit else frame_count or limit
        index = 0
        with tqdm(total=math.ceil(duration / stride) if duration else None, desc="Tracking vehicles") as bar:
            while limit is None or index < limit:
                ok, frame = capture.read()
                if not ok:
                    break
                frame_number = index
                index += 1
                if frame_number % stride:
                    continue
                h, w = frame.shape[:2]
                scale = min(1, 1280 / max(w, h))
                size = (max(2, round(w * scale / 2) * 2), max(2, round(h * scale / 2) * 2))
                if frame.shape[1::-1] != size:
                    frame = cv2.resize(frame, size, interpolation=cv2.INTER_AREA)
                w, h = size
                if writer is None:
                    writer = cv2.VideoWriter(str(output), cv2.VideoWriter_fourcc(*"mp4v"), fps / stride, size)
                    if not writer.isOpened():
                        raise RuntimeError("Could not write the annotated output video.")
                results = model.track(frame, persist=True, tracker="bytetrack.yaml",
                                      classes=list(CLASSES), device=device, imgsz=960, verbose=False)
                boxes = results[0].boxes
                if boxes is not None and boxes.id is not None:
                    observations = zip(boxes.xyxy.cpu().tolist(), boxes.id.int().cpu().tolist(),
                                       boxes.cls.int().cpu().tolist())
                    for (x1, y1, x2, y2), track_id, class_id in observations:
                        label = CLASSES[class_id]
                        estimator.observe(track_id, label, (x1+x2)/(2*(w-1)),
                                          y2/(h-1), frame_number/fps)
                        color = (110, 235, 95) if track_id in estimator.counted else (250, 190, 80)
                        cv2.rectangle(frame, (int(x1), int(y1)), (int(x2), int(y2)), color, 2)
                        cv2.putText(frame, f"{label} #{track_id}",
                                    (int(x1), max(15, int(y1)-4)), FONT, 0.46, color, 1)
                draw_gates(frame)
                title = "Demo estimate" if USE_DEMO_VIDEO else "Estimated tailpipe CO2"
                cv2.rectangle(frame, (0, 0), (min(w-1, 570), 34), (25,25,25), -1)
                cv2.putText(frame, f"{title}: {estimator.total_g/1000:.3f} kg | "
                            f"{len(estimator.events)} traversals", (8,24), FONT, 0.57,
                            (255,255,255), 1, cv2.LINE_AA)
                writer.write(frame)
                bar.update()
        if writer is None:
            raise RuntimeError("No frames were processed.")
        return index / fps
    finally:
        capture.release()
        if writer is not None:
            writer.release()

run_dir = Path(tempfile.mkdtemp(prefix="run-", dir=ROOT))
raw_video = run_dir / "tracked_raw.mp4"
annotated_video = run_dir / "annotated_traffic.mp4"
estimator = SegmentEstimator(GATES, (GATE_X_MIN, GATE_X_MAX), segment_metres, factors)
processed_seconds = run_video(video_path, raw_video, estimator, MAX_VIDEO_SECONDS)
ffmpeg = shutil.which("ffmpeg")
if ffmpeg:
    encode = subprocess.run(
        [ffmpeg, "-v", "error", "-y", "-i", str(raw_video), "-an", "-c:v", "libx264",
         "-pix_fmt", "yuv420p", "-movflags", "+faststart", str(annotated_video)],
        capture_output=True, text=True)
    if encode.returncode == 0:
        raw_video.unlink()
if raw_video.exists():
    raw_video.replace(annotated_video)
    print("The ZIP contains an MP4 that may require an external player.")

In [ ]:
# @title 6. Show and download results
import json
from zipfile import ZipFile
import pandas as pd

columns = ["track_id", "vehicle_class", "direction", "entry_seconds", "exit_seconds",
           "segment_metres", "factor_g_co2_per_km", "estimated_co2_g"]
events = pd.DataFrame(estimator.events, columns=columns)
rows = []
for label, factor in factors.items():
    group = events.loc[events["vehicle_class"] == label]
    rows.append({
        "vehicle_class": label, "A_to_B": int((group["direction"] == "A_to_B").sum()),
        "B_to_A": int((group["direction"] == "B_to_A").sum()),
        "completed_traversals": len(group), "factor_g_co2_per_km": factor,
        "estimated_co2_g": len(group) * segment_metres / 1000 * factor,
    })
totals = pd.DataFrame(rows)
summary = {
    "method": "vehicle class emission factor times completed segment distance",
    "status": "estimated" if len(events) else "no_completed_traversals_detected",
    "mode": "illustrative_demo" if USE_DEMO_VIDEO else "custom_video_estimate",
    "input_filename": video_path.name,
    "processed_seconds": round(processed_seconds, 3), "segment_metres": segment_metres,
    "gate_y": GATES, "gate_x_limits": [GATE_X_MIN, GATE_X_MAX],
    "factor_source": factor_source, "completed_traversals": len(events),
    "estimated_tailpipe_co2_kg": round(estimator.total_g / 1000, 6) if len(events) else None,
    "class_totals": rows,
}
events.to_csv(run_dir / "vehicle_events.csv", index=False)
totals.to_csv(run_dir / "class_totals.csv", index=False)
(run_dir / "summary.json").write_text(json.dumps(summary, indent=2), encoding="utf-8")
bundle = run_dir / "zenith_traffic_co2_results.zip"
with ZipFile(bundle, "w") as archive:
    for name in ("annotated_traffic.mp4", "vehicle_events.csv", "class_totals.csv", "summary.json"):
        archive.write(run_dir / name, name)
if len(events):
    heading = "Illustrative estimate" if USE_DEMO_VIDEO else "Estimated tailpipe CO₂"
    print(f"{heading}: {summary['estimated_tailpipe_co2_kg']:.4f} kg "
          f"from {len(events)} traversals in {processed_seconds:.1f} s.")
else:
    print("No completed traversals detected. Check the gates and tracking in the annotated video.")
display(totals.round(3))
if annotated_video.stat().st_size <= 20_000_000:
    display(Video(str(annotated_video), embed=True, width=780))
print(f"Report ZIP: {bundle}")
if AUTO_DOWNLOAD:
    from google.colab import files
    files.download(str(bundle))

## Interpreting the estimate

Compare the class counts with a manual tally of vehicles that cross both gates. Short clips, occlusion, and tracker ID changes can miss traversals. Broad vehicle classes can also miss or misclassify auto-rickshaws. Local fuel types, fleet composition, and EV share affect the class-average factors. This distance-based estimate omits extra idling and acceleration effects. Validate CO₂ separately using an independent reference such as fuel-use or instrumented emissions data.

[VisualAI_Emission](https://github.com/SonghuaHu-UMD/VisualAI_Emission/blob/master/README.md) is a reference for camera-to-traffic processing and class-count validation. Its citywide DTALite and MOVES-Matrix stages require additional data and software. For a speed-aware upgrade, calibrate the camera to road distances, recover vehicle speed over time, and evaluate [SUMO emissionsDrivingCycle](https://sumo.dlr.de/docs/Tools/Emissions.html#emissionsdrivingcycle) with suitable fleet assumptions.